<a href="https://colab.research.google.com/github/alaaguedda/medical_report_summarization_project/blob/trained/medical_summerizer_trained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d aminexdr/bhc-mimic-iv-summary
!unzip bhc-mimic-iv-summary.zip


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/aminexdr/bhc-mimic-iv-summary
License(s): unknown
100% 444M/446M [00:02<00:00, 102MB/s]
100% 446M/446M [00:02<00:00, 165MB/s]
Archive:  bhc-mimic-iv-summary.zip
  inflating: BHC_MIMIC-IV.csv        


In [2]:
import pandas as pd

df = pd.read_csv("BHC_MIMIC-IV.csv")


In [3]:
df = df[['input', 'target']].dropna()

# Shuffle first
df = df.sample(frac=1, random_state=42)

# Take only 10,000 samples
df = df.iloc[:10000]

df.shape

(10000, 2)

In [4]:
df.head(
)

,input,target
267505,write a discharge summary:\nHistory of Present...,Patient arrived on the unit intubated and seda...
180880,generate a brief hospital summary:\nChief Comp...,"AP: year old female with ho CLL, PAF, not on ..."
252848,create a summary based on the following inform...,Mrs. is a yr old female presenting with new ...
112160,write a discharge summary:\nChief Complaint: C...,Patient had recurring chest pain consistent wi...
99808,create a summary based on the following inform...,"Mr. is a M w ho CAD , atrial fibrillation sp..."


In [5]:
df.isnull().sum()
df = df.dropna()

In [6]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

In [7]:
!pip install transformers datasets evaluate rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=30c0a8fce2a4e4be466b1e1e60856ceb792a40aa30144cc431faf59ef1a68b9f
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [8]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [9]:
max_input_length = 256
max_target_length = 64

def preprocess_function(examples):
    inputs = ["summarize: " + doc for doc in examples["input"]]

    model_inputs = tokenizer(
        inputs,
        max_length=256,
        truncation=True
    )

    labels = tokenizer(
        examples["target"],
        max_length=64,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [10]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [11]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [12]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=2,
    eval_accumulation_steps=10,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    weight_decay=0.01,
    fp16=True,
    predict_with_generate=True,  # Now this will work!
    generation_max_length=64,
    generation_num_beams=1,
    logging_steps=200,
    report_to="none"
)

In [14]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,6.841641,3.162276


TrainOutput(global_step=500, training_loss=6.947536010742187, metrics={'train_runtime': 148.1278, 'train_samples_per_second': 54.007, 'train_steps_per_second': 3.375, 'total_flos': 541367205888000.0, 'train_loss': 6.947536010742187, 'epoch': 1.0})

In [16]:
!pip install evaluate rouge_score absl-py

In [25]:
import torch
import gc

# 1. Clear Python garbage collector
gc.collect()

# 2. Clear NVIDIA cache
torch.cuda.empty_cache()



In [28]:
import numpy as np
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    # Fix: clip predictions to valid token id range
    vocab_size = tokenizer.vocab_size
    predictions = np.clip(predictions, 0, vocab_size - 1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    # Fix: also clip labels just in case
    labels = np.clip(labels, 0, vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    rouge_result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {
        "ROUGE-1": rouge_result["rouge1"],
        "ROUGE-2": rouge_result["rouge2"],
        "ROUGE-L": rouge_result["rougeL"],
    }

In [29]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,  # Fix: renamed from 'tokenizer'
    compute_metrics=compute_metrics
)

In [30]:
results = trainer.evaluate()

In [33]:
print(results)

{'eval_loss': 3.162275791168213, 'eval_model_preparation_time': 0.0027, 'eval_ROUGE-1': 0.34152166183779276, 'eval_ROUGE-2': 0.19069511401749073, 'eval_ROUGE-L': 0.2912175678943819, 'eval_runtime': 443.9552, 'eval_samples_per_second': 2.252, 'eval_steps_per_second': 1.126}


In [37]:
import torch
import textwrap

def generate_and_compare(sample_input, reference_summary=None,
                         input_preview_chars=500,
                         max_input_length=256,
                         max_target_length=64,
                         num_beams=4):
    """
    Generates summary and prints structured comparison.

    Parameters
    ----------
    sample_input : str
        The original medical report text.
    reference_summary : str, optional
        The ground truth summary (if available).
    input_preview_chars : int
        Number of characters from original input to display.
    max_input_length : int
        Token truncation length for input.
    max_target_length : int
        Maximum generation length.
    num_beams : int
        Beam search width.
    """

    device = model.device
    model.eval()

    # ---- Prepare Input ----
    input_text = "summarize: " + sample_input

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=max_input_length,
        truncation=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    # ---- Generate ----
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_target_length,
            num_beams=num_beams,
            early_stopping=True
        )

    generated_summary = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # ---- Pretty Printing Section ----
    print("\n" + "="*80)
    print("📝 ORIGINAL INPUT (Preview)")
    print("="*80)
    print(textwrap.fill(sample_input[:input_preview_chars], width=100))

    print("\n" + "-"*80)
    print("🤖 GENERATED SUMMARY")
    print("-"*80)
    print(textwrap.fill(generated_summary, width=100))


    print("="*80 + "\n")

    return generated_summary

In [39]:
generate_and_compare(
    sample_input=df['input'].iloc[0],
)


📝 ORIGINAL INPUT (Preview)
write a discharge summary: History of Present Illness: The patients oncologic history began in , at
which time he felt a pain in his left side that was initially attributed to musculoskeletal strain.
However, this pain did not remit with over-the-counter analgesics, and a physician in  an abdominal
CT scan. According to the patient and his wife  , this revealed lymphadenopathy suspicious for
lymphoma. He then underwent upper endoscopy with biopsy of a gastric lesion in approximately , which
repo

--------------------------------------------------------------------------------
🤖 GENERATED SUMMARY
--------------------------------------------------------------------------------
Oncologic history began in, at which time he felt a pain in his left side that was initially
attributed to musculoskeletal strain. However, this pain did not remit with over-the-counter
analgesics, and a physician in an abdominal CT



'Oncologic history began in, at which time he felt a pain in his left side that was initially attributed to musculoskeletal strain. However, this pain did not remit with over-the-counter analgesics, and a physician in an abdominal CT'

In [40]:
test_report = """
Patient presents with a history of chronic acute subacute intermittent persistent recurrent episodes of generalized localized diffuse focal systemic symptoms...
"""

generate_and_compare(sample_input=test_report)


📝 ORIGINAL INPUT (Preview)
 Patient presents with a history of chronic acute subacute intermittent persistent recurrent
episodes of generalized localized diffuse focal systemic symptoms...

--------------------------------------------------------------------------------
🤖 GENERATED SUMMARY
--------------------------------------------------------------------------------
Patient presents with a history of chronic acute subacute intermittent persistent recurrent episodes
of generalized localized diffuse focal systemic symptoms...



'Patient presents with a history of chronic acute subacute intermittent persistent recurrent episodes of generalized localized diffuse focal systemic symptoms...'